(history books user đã rate trước đó, candidate book, user features, candidate features) -> label

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mkechinov/ecommerce-behavior-data-from-multi-category-store")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm


def _log_step(step_no: int, msg: str):
    print(f"\n[Step {step_no}] {msg}")


def sample_recsys_subset(
    input_csv: str,
    output_csv: str,
    target_rows: int = 2_000_000,
    min_user_events: int = 5,
    min_user_unique_items: int = 2,
    require_purchase_ratio: float = 0.35,
    top_n_categories: int | None = 30,
    random_state: int = 42,
):
    """
    Create a smaller recsys-friendly subset from Kaggle eCommerce 2019-Oct.csv.

    Strategy:
    - keep rows with valid user/session/product/timestamp
    - optional: keep only top-N category_code
    - build user stats
    - stratified sample users so we preserve:
        * users with purchase
        * users with cart but no purchase
        * browse-only users
    - keep ALL events of sampled users
    - trim near target_rows
    """

    rng = np.random.default_rng(random_state)

    _log_step(1, "Reading CSV")
    usecols = [
        "event_time",
        "event_type",
        "product_id",
        "category_code",
        "brand",
        "price",
        "user_id",
        "user_session",
    ]
    df = pd.read_csv(input_csv, usecols=usecols)
    print(f"Loaded rows={len(df):,}, cols={len(df.columns)}")

    _log_step(2, "Basic cleaning")
    before = len(df)
    df = df.dropna(subset=["event_time", "event_type", "product_id", "user_id", "user_session"])
    print(f"After dropna required cols: {len(df):,} rows (removed {before - len(df):,})")

    before = len(df)
    df = df[df["price"].fillna(0) >= 0]
    print(f"After price filter: {len(df):,} rows (removed {before - len(df):,})")

    df["event_time"] = pd.to_datetime(df["event_time"], utc=True, errors="coerce")
    before = len(df)
    df = df.dropna(subset=["event_time"])
    print(f"After datetime parse/drop invalid time: {len(df):,} rows (removed {before - len(df):,})")

    before = len(df)
    df = df.dropna(subset=["category_code"])
    print(f"After dropna category_code: {len(df):,} rows (removed {before - len(df):,})")

    valid_events = {"view", "cart", "remove_from_cart", "purchase"}
    before = len(df)
    df = df[df["event_type"].isin(valid_events)]
    print(f"After valid event filter: {len(df):,} rows (removed {before - len(df):,})")
    print("Event distribution:")
    print(df["event_type"].value_counts(dropna=False).to_dict())

    _log_step(3, "Optional top category filtering")
    if top_n_categories is not None:
        top_cats = df["category_code"].value_counts().head(top_n_categories).index
        before = len(df)
        df = df[df["category_code"].isin(top_cats)]
        print(
            f"Kept top {top_n_categories} categories -> {len(df):,} rows "
            f"(removed {before - len(df):,}), categories={df['category_code'].nunique():,}"
        )
    else:
        print("Skipped top category filtering")

    _log_step(4, "Building user-level stats")
    user_stats = (
        df.groupby("user_id")
        .agg(
            n_events=("product_id", "size"),
            n_items=("product_id", "nunique"),
            has_purchase=("event_type", lambda x: int((x == "purchase").any())),
            has_cart=("event_type", lambda x: int((x == "cart").any())),
            has_view=("event_type", lambda x: int((x == "view").any())),
        )
        .reset_index()
    )
    print(f"Built user_stats for {len(user_stats):,} users")

    before_users = len(user_stats)
    user_stats = user_stats[
        (user_stats["n_events"] >= min_user_events)
        & (user_stats["n_items"] >= min_user_unique_items)
    ].copy()
    print(
        f"After user history filter: {len(user_stats):,} users "
        f"(removed {before_users - len(user_stats):,})"
    )

    _log_step(5, "Bucketing users by funnel type")
    user_stats["bucket"] = np.select(
        [
            user_stats["has_purchase"] == 1,
            (user_stats["has_purchase"] == 0) & (user_stats["has_cart"] == 1),
        ],
        [2, 1],
        default=0,
    )
    print("Bucket counts:", user_stats["bucket"].value_counts().sort_index().to_dict())

    avg_events = max(user_stats["n_events"].mean(), 1)
    approx_n_users = int(target_rows / avg_events)
    print(f"Average events/user={avg_events:.2f}")
    print(f"Approx users needed={approx_n_users:,} for target_rows={target_rows:,}")

    desired_ratios = {
        2: require_purchase_ratio,
        1: 0.35,
        0: 1.0 - require_purchase_ratio - 0.35,
    }
    print("Desired ratios:", desired_ratios)

    _log_step(6, "Sampling users by bucket")
    selected_user_ids = []

    for bucket, ratio in tqdm(desired_ratios.items(), desc="Sampling buckets"):
        bucket_users = user_stats[user_stats["bucket"] == bucket]
        print(f"Bucket {bucket}: available users={len(bucket_users):,}, target ratio={ratio:.2f}")

        if len(bucket_users) == 0:
            print(f"Bucket {bucket}: skipped (empty)")
            continue

        n_take = min(len(bucket_users), int(approx_n_users * ratio))
        if n_take <= 0:
            print(f"Bucket {bucket}: skipped (n_take <= 0)")
            continue

        sampled = bucket_users.sample(n=n_take, random_state=random_state)["user_id"].tolist()
        selected_user_ids.extend(sampled)
        print(f"Bucket {bucket}: sampled {n_take:,} users")

    if len(selected_user_ids) < approx_n_users:
        _log_step(7, "Fallback sampling remaining users")
        remaining_users = user_stats[~user_stats["user_id"].isin(selected_user_ids)]
        n_more = min(len(remaining_users), approx_n_users - len(selected_user_ids))
        print(f"Remaining users available={len(remaining_users):,}, need more={n_more:,}")

        if n_more > 0:
            more_ids = remaining_users.sample(n=n_more, random_state=random_state)["user_id"].tolist()
            selected_user_ids.extend(more_ids)
            print(f"Fallback sampled {n_more:,} more users")
    else:
        _log_step(7, "Fallback sampling skipped")

    print(f"Total selected users={len(selected_user_ids):,}")

    _log_step(8, "Keeping all events of sampled users")
    subset = df[df["user_id"].isin(selected_user_ids)].copy()
    print(f"Subset rows before sort={len(subset):,}")

    subset = subset.sort_values(["user_id", "event_time"]).reset_index(drop=True)
    print(f"Subset rows after sort={len(subset):,}")

    _log_step(9, "Optional hard trim by user")
    if len(subset) > target_rows * 1.25:
        print(
            f"Subset too large ({len(subset):,} rows), trimming toward target_rows={target_rows:,}"
        )
        selected_user_ids = pd.Series(selected_user_ids)
        sampled_order = selected_user_ids.sample(frac=1, random_state=random_state).tolist()

        kept_chunks = []
        running_rows = 0

        for uid in tqdm(sampled_order, desc="Trimming users"):
            part = subset[subset["user_id"] == uid]
            if running_rows + len(part) > target_rows and running_rows > target_rows * 0.9:
                print(
                    f"Stopping trim at running_rows={running_rows:,}, "
                    f"next_user_rows={len(part):,}"
                )
                break
            kept_chunks.append(part)
            running_rows += len(part)

        subset = pd.concat(kept_chunks, ignore_index=True)
        subset = subset.sort_values(["user_id", "event_time"]).reset_index(drop=True)
        print(f"Subset rows after trim={len(subset):,}")
    else:
        print(f"Trim skipped: subset rows={len(subset):,} within threshold")

    _log_step(10, "Saving subset")
    subset.to_csv(output_csv, index=False)
    print(f"Saved to: {output_csv}")

    _log_step(11, "Computing report stats")
    stats = {
        "rows": len(subset),
        "users": subset["user_id"].nunique(),
        "items": subset["product_id"].nunique(),
        "sessions": subset["user_session"].nunique(),
        "event_dist": subset["event_type"].value_counts(normalize=True).to_dict(),
        "purchase_users": subset.groupby("user_id")["event_type"].apply(lambda x: (x == "purchase").any()).mean(),
        "categories": subset["category_code"].nunique(),
    }

    print("\nFinal stats:")
    for k, v in stats.items():
        print(f"- {k}: {v}")

    return subset, stats

In [ ]:
import pandas as pd
def load_data(df_path: str) -> pd.DataFrame:
    print(f"Loading data from: {df_path}")
    df = pd.read_csv(df_path)
    return df
df = load_data("./data/2019-Oct.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
subset_df, stats = sample_recsys_subset(
    input_csv="./data/2019-Oct.csv",
    output_csv="2019-Oct-recsys-1m.csv",
    target_rows=1_000_000,     # m có thể thử 500k, 1M, 2M
    min_user_events=5,
    min_user_unique_items=2,
    require_purchase_ratio=0.25,
    top_n_categories=30,
    random_state=42,
)

print(stats)

In [ ]:
data_path = "./data/2019-Oct-recsys-1m.csv"
df = pd.read_csv(data_path)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df['event_type'].unique()

In [ ]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import math


def _log(step, msg):
    print(f"\n[Step {step}] {msg}")


class PopularitySampler:
    def __init__(self, item_counts: dict, alpha: float = 1.0, seed: int = 42):
        self.rng = np.random.default_rng(seed)
        items = sorted(item_counts.keys())
        counts = np.array([item_counts[i] for i in items], dtype=np.float64)
        probs = np.power(counts, alpha)
        probs = probs / probs.sum()
        self.items = np.array(items)
        self.probs = probs

    def sample_unseen(self, seen_items: set, positive_item, k: int):
        negatives = []
        forbid = set(seen_items)
        forbid.add(positive_item)

        allowed_items = [x for x in self.items if x not in forbid]
        if len(allowed_items) == 0:
            return negatives

        while len(negatives) < k:
            sampled = self.rng.choice(
                self.items,
                size=(k - len(negatives)) * 3,
                replace=True,
                p=self.probs,
            )
            for item in sampled:
                if item not in forbid:
                    negatives.append(item)
                if len(negatives) == k:
                    break
        return negatives


def process_user_chunk(
    chunk_df: pd.DataFrame,
    item_meta: dict,
    item_counts: dict,
    min_history_len: int,
    max_history_len: int,
    num_negatives: int,
    neg_alpha: float,
    positive_events: tuple,
    history_events: tuple,
    random_state: int,
):
    sampler = PopularitySampler(
        item_counts=item_counts,
        alpha=neg_alpha,
        seed=random_state,
    )

    rows = []

    grouped = chunk_df.groupby("user_id", sort=False)
    for user_id, g in grouped:
        g = g.sort_values("event_time").reset_index(drop=True)

        hist_items = []
        hist_event_types = []
        hist_categories = []
        hist_brands = []
        hist_price_buckets = []
        hist_times = []
        seen_items = set()

        for i in range(len(g)):
            row = g.iloc[i]
            cur_item = row["product_id"]
            cur_event = row["event_type"]

            if cur_event in positive_events and len(hist_items) >= min_history_len:
                cur_hist_items = hist_items[-max_history_len:]
                cur_hist_event_types = hist_event_types[-max_history_len:]
                cur_hist_categories = hist_categories[-max_history_len:]
                cur_hist_brands = hist_brands[-max_history_len:]
                cur_hist_price_buckets = hist_price_buckets[-max_history_len:]
                cur_hist_times = hist_times[-max_history_len:]

                rows.append({
                    "user_id": user_id,
                    "event_time": row["event_time"],
                    "label": 1,
                    "target_item_id": cur_item,
                    "target_category": row["category_code"],
                    "target_brand": row["brand"],
                    "target_price": row["price"],
                    "target_price_bucket": row["price_bucket"],
                    "target_event_type": cur_event,
                    "target_event_type_id": row["event_type_id"],
                    "hist_item_id": cur_hist_items.copy(),
                    "hist_event_type": cur_hist_event_types.copy(),
                    "hist_category": cur_hist_categories.copy(),
                    "hist_brand": cur_hist_brands.copy(),
                    "hist_price_bucket": cur_hist_price_buckets.copy(),
                    "hist_time": cur_hist_times.copy(),
                    "hist_len": len(cur_hist_items),
                })

                neg_items = sampler.sample_unseen(
                    seen_items=seen_items,
                    positive_item=cur_item,
                    k=num_negatives,
                )

                for neg_item in neg_items:
                    neg_meta = item_meta[neg_item]
                    rows.append({
                        "user_id": user_id,
                        "event_time": row["event_time"],
                        "label": 0,
                        "target_item_id": neg_item,
                        "target_category": neg_meta["category_code"],
                        "target_brand": neg_meta["brand"],
                        "target_price": neg_meta["price"],
                        "target_price_bucket": neg_meta["price_bucket"],
                        "target_event_type": "NEGATIVE",
                        "target_event_type_id": 0,
                        "hist_item_id": cur_hist_items.copy(),
                        "hist_event_type": cur_hist_event_types.copy(),
                        "hist_category": cur_hist_categories.copy(),
                        "hist_brand": cur_hist_brands.copy(),
                        "hist_price_bucket": cur_hist_price_buckets.copy(),
                        "hist_time": cur_hist_times.copy(),
                        "hist_len": len(cur_hist_items),
                    })

            if cur_event in history_events:
                hist_items.append(cur_item)
                hist_event_types.append(row["event_type_id"])
                hist_categories.append(row["category_code"])
                hist_brands.append(row["brand"])
                hist_price_buckets.append(row["price_bucket"])
                hist_times.append(row["event_time"])
                seen_items.add(cur_item)

    return rows


def split_user_chunks(user_ids, n_chunks):
    chunk_size = math.ceil(len(user_ids) / n_chunks)
    return [user_ids[i:i + chunk_size] for i in range(0, len(user_ids), chunk_size)]


def build_bst_ranking_dataset_mp(
    input_csv: str,
    output_jsonl: str,
    min_history_len: int = 2,
    max_history_len: int = 20,
    num_negatives: int = 3,
    neg_alpha: float = 0.75,
    positive_events=("cart", "purchase"),
    history_events=("view", "cart", "purchase"),
    random_state: int = 42,
    num_workers: int = 4,
):
    _log(1, "Reading sampled recsys CSV")
    df = pd.read_csv(input_csv)
    print(f"Loaded rows={len(df):,}")

    _log(2, "Basic preprocessing")
    df["event_time"] = pd.to_datetime(df["event_time"], utc=True, errors="coerce")
    df = df.dropna(subset=["event_time", "user_id", "product_id", "event_type"])
    df = df.sort_values(["user_id", "event_time"]).reset_index(drop=True)

    df["category_code"] = df["category_code"].fillna("UNK_CATEGORY")
    df["brand"] = df["brand"].fillna("UNK_BRAND")
    df["price"] = df["price"].fillna(0.0)

    df["price_bucket"] = pd.qcut(
        df["price"].rank(method="first"),
        q=min(10, df["price"].nunique()),
        labels=False,
        duplicates="drop",
    )
    df["price_bucket"] = df["price_bucket"].fillna(0).astype(int)

    event2id = {e: i + 1 for i, e in enumerate(sorted(df["event_type"].unique()))}
    df["event_type_id"] = df["event_type"].map(event2id)

    print(f"Users={df['user_id'].nunique():,}, Items={df['product_id'].nunique():,}")
    print(f"Event types={event2id}")

    _log(3, "Building item metadata and popularity counts")
    item_counts = df["product_id"].value_counts().to_dict()

    item_meta = (
        df.sort_values("event_time")
        .groupby("product_id", as_index=False)
        .first()[["product_id", "category_code", "brand", "price", "price_bucket"]]
        .set_index("product_id")
        .to_dict("index")
    )

    user_ids = df["user_id"].drop_duplicates().tolist()
    chunks = split_user_chunks(user_ids, num_workers)
    print(f"Total users={len(user_ids):,}, chunks={len(chunks):,}, workers={num_workers}")

    _log(4, "Multiprocessing user chunks")
    all_rows = []

    with ProcessPoolExecutor(max_workers=num_workers) as ex:
        futures = []
        for idx, chunk_user_ids in enumerate(chunks):
            chunk_df = df[df["user_id"].isin(chunk_user_ids)].copy()
            futures.append(
                ex.submit(
                    process_user_chunk,
                    chunk_df,
                    item_meta,
                    item_counts,
                    min_history_len,
                    max_history_len,
                    num_negatives,
                    neg_alpha,
                    positive_events,
                    history_events,
                    random_state + idx,
                )
            )

        for fut in tqdm(as_completed(futures), total=len(futures), desc="Finished chunks"):
            all_rows.extend(fut.result())

    _log(5, "Building final dataframe")
    bst_df = pd.DataFrame(all_rows)
    print(f"Built BST rows={len(bst_df):,}")

    _log(6, "Saving")
    bst_df = bst_df.sort_values(["user_id", "event_time", "label"], ascending=[True, True, False]).reset_index(drop=True)
    bst_df.to_json(output_jsonl, orient="records", lines=True)
    print(f"Saved to {output_jsonl}")

    stats = {
        "rows": len(bst_df),
        "users": bst_df["user_id"].nunique(),
        "positive_rows": int((bst_df["label"] == 1).sum()),
        "negative_rows": int((bst_df["label"] == 0).sum()),
        "avg_hist_len": float(bst_df["hist_len"].mean()) if len(bst_df) else 0.0,
        "target_items": bst_df["target_item_id"].nunique(),
    }

    _log(7, "Final stats")
    for k, v in stats.items():
        print(f"- {k}: {v}")

    return bst_df, stats

In [ ]:
bst_df, stats = build_bst_ranking_dataset_mp(
    input_csv="./data/2019-Oct-recsys-1m.csv",
    output_jsonl="./data/2019-Oct-bst-ranking-1m.json",
    min_history_len=2,
    max_history_len=20,
    num_negatives=3,
    neg_alpha=0.75,
    positive_events=("cart", "purchase"),
    history_events=("view", "cart", "purchase"),
    random_state=42,
)

# Feature Engineering

In [ ]:
import pandas as pd
path = './data/2019-Oct-bst-ranking-1m.csv'
df = pd.read_csv(path)

In [ ]:
df.info()

In [ ]:
# other features : target_price, target_price_bucket
# user behavior features: hist_event_type, hist_category, hist_brand, hist_price_bucket, hist_time
# target item features: target_category, target_brand, target_price_bucket, event_time